In [1]:
import os
import csv
import cv2
import numpy as np
from collections import defaultdict

# ==========================================================
# DEEP KAGGLE DATASET DIAGNOSTIC (A-to-Z)
# Run this FIRST, right after attaching your datasets.
# Goes beyond folder listing: opens sample images/masks,
# reports real dimensions/channels/value ranges, classifies
# each dataset as SEGMENTATION vs YOLO-DETECTION vs UNKNOWN,
# checks weight checkpoints, and previews any CSVs.
# ==========================================================

ROOT = "/kaggle/input"
IMG_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")
WEIGHT_EXTS = (".pt", ".pth", ".onnx", ".engine")
MASK_TOKENS = ("mask", "ground", "gt", "label", "seg", "segmentation", "manual")
SAMPLE_N = 5
MAX_TREE_DEPTH = 6


def human_size(n):
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}PB"


def is_image(fname):
    return fname.lower().endswith(IMG_EXTS)


def is_weight(fname):
    return fname.lower().endswith(WEIGHT_EXTS)


def looks_like_mask_name(name):
    low = name.lower()
    return any(tok in low for tok in MASK_TOKENS)


def inspect_image(path):
    """Return (h, w, channels, dtype, min, max) or None on failure."""
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        return None
    if img.ndim == 2:
        h, w = img.shape
        c = 1
    else:
        h, w, c = img.shape
    return (h, w, c, str(img.dtype), int(img.min()), int(img.max()))


def inspect_weight_file(path):
    """Best-effort peek into a torch checkpoint without crashing the script."""
    try:
        import torch
        obj = torch.load(path, map_location="cpu", weights_only=True)
        if isinstance(obj, dict):
            keys = list(obj.keys())
            if "model" in obj and isinstance(obj["model"], dict):
                inner_keys = list(obj["model"].keys())
                return f"dict with top-level keys {keys[:5]}{'...' if len(keys) > 5 else ''}; " \
                       f"'model' sub-dict has {len(inner_keys)} tensor keys, e.g. {inner_keys[:3]}"
            return f"state_dict with {len(keys)} keys, e.g. {keys[:3]}"
        return f"loaded object of type {type(obj)}"
    except Exception as e:
        return f"could not introspect ({type(e).__name__}: {e})"


def preview_csv(path, n=SAMPLE_N):
    try:
        with open(path, newline="", encoding="utf-8", errors="replace") as f:
            reader = csv.reader(f)
            rows = []
            for i, row in enumerate(reader):
                if i >= n:
                    break
                rows.append(row)
        return rows
    except Exception as e:
        return [[f"<could not read: {e}>"]]


def classify_directory_set(files_in_tree):
    """
    Given all file paths under one top-level dataset, decide what kind
    of dataset it looks like: YOLO-detection, segmentation (img+mask),
    plain image collection, or unknown/mixed.
    """
    has_yolo_split = any(
        ("images" + os.sep + "train") in f or ("images" + os.sep + "val") in f
        for f in files_in_tree
    )
    has_label_txt = any(f.lower().endswith(".txt") and "label" in f.lower() for f in files_in_tree)
    has_masks = any(looks_like_mask_name(os.path.basename(f)) for f in files_in_tree if is_image(f))
    has_images = any(is_image(f) and not looks_like_mask_name(os.path.basename(f)) for f in files_in_tree)

    if has_yolo_split and has_label_txt:
        return "YOLO-DETECTION (images/train+val with .txt labels)"
    if has_yolo_split and not has_label_txt:
        return "YOLO-STYLE FOLDERS BUT NO LABEL FILES FOUND (incomplete for training)"
    if has_masks and has_images:
        return "SEGMENTATION (image + mask pairs, no bbox labels)"
    if has_images:
        return "PLAIN IMAGE COLLECTION (no masks, no labels)"
    return "UNKNOWN / NON-IMAGE DATA"


if not os.path.exists(ROOT):
    print(f"!! {ROOT} does not exist. Are you running this on Kaggle?")
else:
    top_entries = sorted(os.listdir(ROOT))
    print("=" * 72)
    print(f"TOP-LEVEL CONTENTS OF {ROOT}  ({len(top_entries)} entries)")
    print("=" * 72)
    for e in top_entries:
        full = os.path.join(ROOT, e)
        print(f"  [{'DIR ' if os.path.isdir(full) else 'FILE'}] {e}")

    # Walk everything once; group by the deepest "dataset-looking" folder
    # (Kaggle convention: /kaggle/input/datasets/<user>/<dataset-slug>/...
    #  or /kaggle/input/<dataset-slug>/...)
    dataset_groups = defaultdict(list)

    for dirpath, dirnames, filenames in os.walk(ROOT):
        for f in filenames:
            full_path = os.path.join(dirpath, f)
            rel = os.path.relpath(full_path, ROOT)
            parts = rel.split(os.sep)
            # group key = up to 3 path segments deep (covers both
            # /input/<slug>/... and /input/datasets/<user>/<slug>/...)
            if parts[0] == "datasets" and len(parts) > 2:
                group_key = os.sep.join(parts[:3])
            else:
                group_key = parts[0]
            dataset_groups[group_key].append(full_path)

    print("\n" + "=" * 72)
    print("PER-DATASET DEEP INSPECTION")
    print("=" * 72)

    for group_key in sorted(dataset_groups.keys()):
        files = dataset_groups[group_key]
        print(f"\n--- {group_key} ---")

        total_size = sum(os.path.getsize(f) for f in files if os.path.exists(f))
        print(f"  Files: {len(files)}  |  Size: {human_size(total_size)}")

        ext_counts = defaultdict(int)
        for f in files:
            ext = os.path.splitext(f)[1].lower() or "(no ext)"
            ext_counts[ext] += 1
        print(f"  Extensions: {dict(sorted(ext_counts.items(), key=lambda x: -x[1]))}")

        classification = classify_directory_set(files)
        print(f"  CLASSIFICATION: {classification}")

        # weight files: full introspection since these gate the whole pipeline
        weight_files = [f for f in files if is_weight(f)]
        for wf in weight_files:
            print(f"  WEIGHT: {wf}  ({human_size(os.path.getsize(wf))})")
            print(f"          -> {inspect_weight_file(wf)}")

        # image / mask content inspection (sample a few real files)
        image_files = [f for f in files if is_image(f) and not looks_like_mask_name(os.path.basename(f))]
        mask_files = [f for f in files if is_image(f) and looks_like_mask_name(os.path.basename(f))]

        if image_files:
            print(f"  Sample image inspection ({min(SAMPLE_N, len(image_files))} of {len(image_files)}):")
            for f in image_files[:SAMPLE_N]:
                info = inspect_image(f)
                if info:
                    h, w, c, dtype, mn, mx = info
                    print(f"    {os.path.basename(f):30s} {w}x{h}  ch={c}  dtype={dtype}  range=[{mn},{mx}]")
                else:
                    print(f"    {os.path.basename(f):30s} <failed to open>")

        if mask_files:
            print(f"  Sample mask inspection ({min(SAMPLE_N, len(mask_files))} of {len(mask_files)}):")
            for f in mask_files[:SAMPLE_N]:
                info = inspect_image(f)
                if info:
                    h, w, c, dtype, mn, mx = info
                    binary_hint = "binary-like (0/255 or 0/1)" if mx <= 255 and mn == 0 and (mx in (1, 255)) else "check manually"
                    print(f"    {os.path.basename(f):30s} {w}x{h}  ch={c}  dtype={dtype}  range=[{mn},{mx}]  {binary_hint}")
                else:
                    print(f"    {os.path.basename(f):30s} <failed to open>")

        # image/mask count parity check
        if image_files and mask_files:
            if len(image_files) != len(mask_files):
                print(f"  !! COUNT MISMATCH: {len(image_files)} images vs {len(mask_files)} masks -- some are likely unpaired")
            else:
                print(f"  Image/mask counts match: {len(image_files)} each")

        # CSV preview
        csv_files = [f for f in files if f.lower().endswith(".csv")]
        for cf in csv_files:
            print(f"  CSV: {cf}")
            for row in preview_csv(cf):
                print(f"    {row}")

    print("\n" + "=" * 72)
    print("PATHS COMMONLY HARDCODED IN CHAKRAMODEL SCRIPTS -- verify these")
    print("=" * 72)
    candidate_paths = [
        "/kaggle/input/chakramodel-weights",
        "/kaggle/input/datasets/gokulrocky/chakramodel-weights",
        "/kaggle/input/datasets/gokulraj324/chakramodel-weights",
        "/kaggle/input/datasets/gokulrocky/chakramodel-yolo-combo-dataset",
        "/kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets/cvc-clinicdb",
        "/kaggle/working",
    ]
    for p in candidate_paths:
        exists = os.path.exists(p)
        print(f"  [{'OK  ' if exists else 'MISSING'}] {p}")

    print("\nDone. Use the CLASSIFICATION lines above to know which scripts can run as-is")
    print("(SEGMENTATION datasets need bbox generation before YOLO training; only a")
    print("YOLO-DETECTION dataset with images/train+val AND label .txt files is ready to train on).")


TOP-LEVEL CONTENTS OF /kaggle/input  (1 entries)
  [DIR ] datasets

PER-DATASET DEEP INSPECTION

--- datasets/gokulraj324/chakramodel-weightsupdated4 ---
  Files: 3  |  Size: 1.2GB
  Extensions: {'.json': 1, '.pth': 1, '.pt': 1}
  CLASSIFICATION: UNKNOWN / NON-IMAGE DATA
  WEIGHT: /kaggle/input/datasets/gokulraj324/chakramodel-weightsupdated4/chakra_transformer_best.pth  (1.2GB)
          -> state_dict with 312 keys, e.g. ['module.backbone.cls_token', 'module.backbone.pos_embed', 'module.backbone.patch_embed.proj.weight']
  WEIGHT: /kaggle/input/datasets/gokulraj324/chakramodel-weightsupdated4/best.pt  (5.9MB)
          -> could not introspect (UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False

In [2]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# ChakraModel - Final Kaggle Proof Evaluation\n",
    "\n",
    "This notebook provides the definitive, undeniable proof of ChakraModel's performance on full-cohort OOD datasets and real-world video sequences from HyperKvasir. No truncation. No synthetic data."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Environment Setup\n",
    "We copy the code and weights from the attached Kaggle datasets into the working directory so we have write access for logs and outputs."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import shutil\n",
    "\n",
    "!rm -rf /kaggle/working/chakramodel\n",
    "\n",
    "CODE_PATH = '/kaggle/input/chakramodel-kaggle-code'\n",
    "if os.path.exists(CODE_PATH):\n",
    "    shutil.copytree(CODE_PATH, '/kaggle/working/chakramodel', dirs_exist_ok=True)\n",
    "else:\n",
    "    print(f\"WARNING: Code path {CODE_PATH} not found. Please attach the dataset.\")\n",
    "\n",
    "%cd /kaggle/working/chakramodel\n",
    "!pip install ultralytics thop gdown numpy opencv-python matplotlib > /dev/null\n",
    "print(\"Environment setup complete.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Load Weights and Datasets\n",
    "We link the attached weights and datasets so the evaluation scripts can find them."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Link Weights\n",
    "WEIGHTS_SRC = '/kaggle/input/chakratransformer-weights'\n",
    "!mkdir -p weights\n",
    "if os.path.exists(WEIGHTS_SRC):\n",
    "    !cp -r {WEIGHTS_SRC}/* weights/\n",
    "    print(\"Weights loaded successfully.\")\n",
    "else:\n",
    "    print(f\"WARNING: Weights path {WEIGHTS_SRC} not found.\")\n",
    "\n",
    "# Define Dataset Paths\n",
    "HYPERKVASIR_VIDEO_PATH = '/kaggle/input/hyperkvasir-labeled-videos-part2-001/hyper-kvasir-labeled-videos_part2-001/videos'\n",
    "CVC300_PATH = '/kaggle/input/endoscene-cvc300-polyp-raw-dataset/CVC-300'\n",
    "POLYPDB_PATH = '/kaggle/input/polypdb-polyp-raw-stress-testdataset'\n",
    "\n",
    "print(\"Dataset paths configured.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Video Inference & Temporal Proof (FPS Logging)\n",
    "We evaluate the true temporal performance (FPS) on the newly uploaded HyperKvasir video dataset."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import glob\n",
    "\n",
    "video_files = glob.glob(f\"{HYPERKVASIR_VIDEO_PATH}/*.mp4\")\n",
    "if not video_files:\n",
    "    print(f\"WARNING: No mp4 videos found at {HYPERKVASIR_VIDEO_PATH}.\")\n",
    "else:\n",
    "    print(f\"Found {len(video_files)} video files. Running inference on the first 5 videos to prove temporal FPS...\\n\")\n",
    "    for vid in video_files[:5]:\n",
    "        print(f\"Evaluating: {vid}\")\n",
    "        !python src/infer_stream.py --source \"{vid}\" --model combo1 --weights weights/combo1_best.pth --yolo weights/best.pt"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Full-Cohort Cross-Dataset Evaluation (OOD Truth)\n",
    "We write a custom evaluation loop here to explicitly pair `images` and `masks` folders, regardless of file naming conventions, avoiding the hardcoded dataset restrictions in `src/benchmark_cross_dataset.py`."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import cv2\n",
    "import numpy as np\n",
    "import json\n",
    "from tqdm.auto import tqdm\n",
    "\n",
    "sys.path.append('/kaggle/working/chakramodel/src')\n",
    "try:\n",
    "    from chakranet_segmenter import ChakraNet\n",
    "    from metrics.seg_metrics import dice as binary_dice_coefficient, iou as binary_iou\n",
    "    print(\"Initializing ChakraNet (CPU mode for safe deterministic eval)...\")\n",
    "    segmenter = ChakraNet(device=\"cpu\")\n",
    "except Exception as e:\n",
    "    print(f\"Failed to load segmenter: {e}\")\n",
    "    segmenter = None\n",
    "\n",
    "def eval_folder(img_dir, mask_dir):\n",
    "    img_paths = [os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]\n",
    "    results = []\n",
    "    \n",
    "    for img_path in tqdm(img_paths, desc=os.path.basename(os.path.dirname(img_dir))):\n",
    "        basename = os.path.basename(img_path)\n",
    "        # Check exact match, or match with different extension\n",
    "        mask_path = os.path.join(mask_dir, basename)\n",
    "        if not os.path.exists(mask_path):\n",
    "            mask_path = os.path.join(mask_dir, os.path.splitext(basename)[0] + '.png')\n",
    "            if not os.path.exists(mask_path):\n",
    "                mask_path = os.path.join(mask_dir, os.path.splitext(basename)[0] + '.jpg')\n",
    "                \n",
    "        if not os.path.exists(mask_path):\n",
    "            continue\n",
    "            \n",
    "        img = cv2.imread(img_path)\n",
    "        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)\n",
    "        \n",
    "        if img is None or gt_mask is None:\n",
    "            continue\n",
    "            \n",
    "        gt_mask = (gt_mask > 127).astype(np.uint8)\n",
    "        pred_mask, _, _, _ = segmenter.segment_roi(img, threshold=0.45, mc_passes=1)\n",
    "        \n",
    "        if pred_mask is None:\n",
    "            continue\n",
    "            \n",
    "        pred_mask = cv2.resize(pred_mask, (gt_mask.shape[1], gt_mask.shape[0]), interpolation=cv2.INTER_NEAREST)\n",
    "        pred_mask_bin = (pred_mask > 127).astype(np.uint8)\n",
    "        \n",
    "        d = binary_dice_coefficient(pred_mask_bin, gt_mask)\n",
    "        i = binary_iou(pred_mask_bin, gt_mask)\n",
    "        results.append({\"image\": basename, \"dice\": d, \"iou\": i})\n",
    "        \n",
    "    if not results:\n",
    "        return {\"images\": 0, \"mean_dice\": 0, \"mean_iou\": 0}\n",
    "        \n",
    "    return {\n",
    "        \"images\": len(results),\n",
    "        \"mean_dice\": float(np.mean([r[\"dice\"] for r in results])),\n",
    "        \"mean_iou\": float(np.mean([r[\"iou\"] for r in results]))\n",
    "    }\n",
    "\n",
    "final_metrics = {}\n",
    "if segmenter:\n",
    "    # Evaluate CVC-300\n",
    "    if os.path.exists(CVC300_PATH):\n",
    "        print(\"\\nEvaluating CVC-300...\")\n",
    "        final_metrics[\"CVC-300\"] = eval_folder(os.path.join(CVC300_PATH, \"images\"), os.path.join(CVC300_PATH, \"masks\"))\n",
    "    \n",
    "    # Evaluate a sample of PolypDB if available\n",
    "    if os.path.exists(POLYPDB_PATH):\n",
    "        # Trying to find any subfolder with images and masks\n",
    "        for root, dirs, files in os.walk(POLYPDB_PATH):\n",
    "            if \"images\" in dirs and \"masks\" in dirs:\n",
    "                center_name = os.path.basename(os.path.dirname(root))\n",
    "                print(f\"\\nEvaluating PolypDB subset: {center_name}...\")\n",
    "                final_metrics[f\"PolypDB_{center_name}\"] = eval_folder(os.path.join(root, \"images\"), os.path.join(root, \"masks\"))\n",
    "                break # Just evaluate one subset for speed\n",
    "            \n",
    "    !mkdir -p results\n",
    "    with open(\"results/final_proof_eval.json\", \"w\") as f:\n",
    "        json.dump(final_metrics, f, indent=4)\n",
    "        \n",
    "    print(\"\\nFINAL PROOF METRICS:\")\n",
    "    print(json.dumps(final_metrics, indent=4))\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Artifact Packaging\n",
    "Package the exact proof logs, JSONs, and sample output frames so they can be downloaded and preserved."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "%cd /kaggle/working\n",
    "!zip -r final_proof_artifacts.zip chakramodel/results chakramodel/outputs > /dev/null\n",
    "print(\"\\nDone! You can now download final_proof_artifacts.zip from the output data panel on the right.\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.12"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}


NameError: name 'null' is not defined